# Sourcing a Model Under Contract

_Investigation `model-sourcing` — coder reproduction notebook._

**Question.** Faced with a modeling task, can an agent SOURCE the model well — reuse an existing module, compose several, or build a new one when justified — and can that sourcing decision be held to a contract that rewards reuse, catches reinvention, and refuses a module that does not actually fit?

This investigation extends the model-build-under-contract loop with a SELECT phase: before the tests are locked, the agent decides WHERE the model comes from. Each of six tasks declares the capabilities it requires; three real modules declare the capabilities they provide; the module_sourcing audit grades the decision on four axes (source_fit, reinvention, novelty_justified, survey_recorded).

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/viva-casebook--composites').is_dir():
    REPO = Path('/Users/eranagmon/code/viva-casebook--composites')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_casebook.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Cell Jostling (`cell-jostling`)

**Question.** Given the task's required capabilities ['physics_2d', 'rigid_body', 'collision'], what is the right way to source the model — reuse, compose, or build-new — and does the sourcing audit confirm it?

**Claim.** The right sourcing for this task is to reuse viva-munk.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva-munk` | `viva_munk.composites.biofilm` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_munk.composites.biofilm`** — `spec_viva_munk_composites_biofilm` (a plain, editable dict)


_composite spec file for `viva_munk.composites.biofilm` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cell-jostling ===
STUDY = 'cell-jostling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Sourcing Audit Report Card**


In [ ]:
# Sourcing Audit Report Card
show_viz(_render_one('local:CellJostlingSourcingAudit', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sourcing-fit | kind=audit-axis path=sourcing.source_fit | op == value within_tol provenance module_sourcing.build_sourcing_report -> source_fit=within_tol |
| no-reinvention | kind=audit-axis path=sourcing.reinvention | op == value within_tol provenance module_sourcing -> reinvention=within_tol |


## Study: Growth And Push (`growth-and-push`)

**Question.** Given the task's required capabilities ['growth', 'physics_2d'], what is the right way to source the model — reuse, compose, or build-new — and does the sourcing audit confirm it?

**Claim.** The right sourcing for this task is to compose growth-proc + viva-munk.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `growth-proc` | `viva_munk.composites.glucose_growth` | 0 | — |
| `viva-munk` | `viva_munk.composites.glucose_growth` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_munk.composites.glucose_growth`** — `spec_viva_munk_composites_glucose_growth` (a plain, editable dict)


_composite spec file for `viva_munk.composites.glucose_growth` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: growth-and-push ===
STUDY = 'growth-and-push'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Sourcing Audit Report Card**


In [ ]:
# Sourcing Audit Report Card
show_viz(_render_one('local:GrowthAndPushSourcingAudit', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sourcing-fit | kind=audit-axis path=sourcing.source_fit | op == value within_tol provenance module_sourcing.build_sourcing_report -> source_fit=within_tol |
| no-reinvention | kind=audit-axis path=sourcing.reinvention | op == value within_tol provenance module_sourcing -> reinvention=within_tol |


## Study: Spatial Competition (`spatial-competition`)

**Question.** Given the task's required capabilities ['spatial', 'dfba', 'diffusion'], what is the right way to source the model — reuse, compose, or build-new — and does the sourcing audit confirm it?

**Claim.** The right sourcing for this task is to reuse spatio-flux.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `spatio-flux` | `spatio_flux.composites.metabolism.community_dfba` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.metabolism.community_dfba`** — `spec_spatio_flux_composites_metabolism_community_dfba` (a plain, editable dict)


_composite spec file for `spatio_flux.composites.metabolism.community_dfba` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spatial-competition ===
STUDY = 'spatial-competition'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Sourcing Audit Report Card**


In [ ]:
# Sourcing Audit Report Card
show_viz(_render_one('local:SpatialCompetitionSourcingAudit', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sourcing-fit | kind=audit-axis path=sourcing.source_fit | op == value within_tol provenance module_sourcing.build_sourcing_report -> source_fit=within_tol |
| no-reinvention | kind=audit-axis path=sourcing.reinvention | op == value within_tol provenance module_sourcing -> reinvention=within_tol |


## Study: Shape Dynamics (`shape-dynamics`)

**Question.** Given the task's required capabilities ['cpm', 'cell_shape', 'morphology'], what is the right way to source the model — reuse, compose, or build-new — and does the sourcing audit confirm it?

**Claim.** The right sourcing for this task is to reuse viva-cpm.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva-cpm` | `viva_casebook.composites.shape-dynamics` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.shape-dynamics`** — `spec_viva_casebook_composites_shape_dynamics` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_casebook_composites_shape_dynamics = load_spec(REPO / 'viva_casebook/composites/shape-dynamics.composite.yaml')
describe_spec(spec_viva_casebook_composites_shape_dynamics)

In [ ]:
# === Edit parameters for composite 'shape-dynamics' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cpm'  (local:CPMProcess)
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['potts']['dims'] = [40, 40, 1]
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['potts']['boundary'] = 'noflux'
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['potts']['neighbor_order'] = 2
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['potts']['temperature'] = 10.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['potts']['seed'] = 1
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][0]['type'] = 1
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][0]['target_volume'] = 80.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][0]['lambda_volume'] = 2.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][0]['target_surface'] = 40.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][0]['lambda_surface'] = 0.5
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][0]['seed_block'] = [8, 8, 0, 14, 14, 1]
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][1]['type'] = 1
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][1]['target_volume'] = 80.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][1]['lambda_volume'] = 2.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][1]['target_surface'] = 40.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][1]['lambda_surface'] = 0.5
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['cells'][1]['seed_block'] = [24, 24, 0, 30, 30, 1]
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['contact'][0]['a'] = 1
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['contact'][0]['b'] = 1
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['spec']['contact'][0]['j'] = 10.0
spec_viva_casebook_composites_shape_dynamics['state']['cpm']['config']['mcs_per_update'] = 10

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: shape-dynamics ===
STUDY = 'shape-dynamics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Shape relaxation toward target volume**


In [ ]:
# Shape relaxation toward target volume
show_viz(_render_one('local:ShapeRelaxation', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sourcing-fit | kind=audit-axis path=sourcing.source_fit | op == value within_tol provenance module_sourcing.build_sourcing_report -> source_fit=within_tol |
| no-reinvention | kind=audit-axis path=sourcing.reinvention | op == value within_tol provenance module_sourcing -> reinvention=within_tol |


## Study: Novel Mechanism (`novel-mechanism`)

**Question.** Given the task's required capabilities ['quantum_signal', 'exotic_transport'], what is the right way to source the model — reuse, compose, or build-new — and does the sourcing audit confirm it?

**Claim.** The right sourcing for this task is to build-new (justified).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `build-new` | `viva_casebook.composites.novel-mechanism` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.novel-mechanism`** — `spec_viva_casebook_composites_novel_mechanism` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_casebook_composites_novel_mechanism = load_spec(REPO / 'viva_casebook/composites/novel-mechanism.composite.yaml')
describe_spec(spec_viva_casebook_composites_novel_mechanism)

In [ ]:
# === Edit parameters for composite 'novel-mechanism' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: novel-mechanism ===
STUDY = 'novel-mechanism'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Sourcing audit report-card (build-new)**


In [ ]:
# Sourcing audit report-card (build-new)
show_viz(_render_one('local:SourcingAuditNovel', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sourcing-fit | kind=audit-axis path=sourcing.source_fit | op == value within_tol provenance module_sourcing.build_sourcing_report -> source_fit=within_tol |
| no-reinvention | kind=audit-axis path=sourcing.reinvention | op == value within_tol provenance module_sourcing -> reinvention=within_tol |


## Study: Trap Wrong Reuse (`trap-wrong-reuse`)

**Question.** Given the task's required capabilities ['physics_2d', 'spatial'], what is the right way to source the model — reuse, compose, or build-new — and does the sourcing audit confirm it?

**Claim.** The right sourcing for this task is to no clean fit — needs `spatial`, which viva-munk lacks.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva-munk` | `viva_munk.composites.biofilm` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_munk.composites.biofilm`** — `spec_viva_munk_composites_biofilm` (a plain, editable dict)


_composite spec file for `viva_munk.composites.biofilm` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: trap-wrong-reuse ===
STUDY = 'trap-wrong-reuse'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Sourcing audit caught the trap**


In [ ]:
# Sourcing audit caught the trap
show_viz(_render_one('local:SourcingAuditTrap', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sourcing-fit | kind=audit-axis path=sourcing.source_fit | op == value within_tol provenance module_sourcing.build_sourcing_report -> source_fit=mismatch |
| no-reinvention | kind=audit-axis path=sourcing.reinvention | op == value within_tol provenance module_sourcing -> reinvention=within_tol |
